**Lab type:** review  
**Course:** NL301 Natural Language Processing with Python  
**Lesson:** 07 — Word and Sentence Embeddings  
**Task:** Review a hybrid retrieval system and answer five questions about embedding behaviour and score comparability.

## Setup

In [ ]:
!pip install sentence-transformers scikit-learn numpy --quiet
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Mixed corpus: some short docs, some long docs
short_docs = [
    "Python error",
    "FastAPI install",
    "pandas merge",
]
long_docs = [
    "When training a machine learning model with scikit-learn, always use a Pipeline to avoid data leakage between preprocessing and cross-validation folds.",
    "Sentence transformers encode variable-length text into fixed-size dense vectors using a pretrained transformer with mean pooling over the final hidden states.",
    "FAISS is a library for efficient similarity search and clustering of dense vectors, developed by Meta AI for billion-scale nearest neighbour retrieval.",
]
corpus = short_docs + long_docs
query  = "how to install FastAPI"


## The system to review

In [ ]:
# Hybrid system under review — read before answering the questions below

# TF-IDF scores for short docs
tv = TfidfVectorizer()
tfidf_matrix = tv.fit_transform(short_docs)
tfidf_query  = tv.transform([query])
tfidf_scores = cosine_similarity(tfidf_query, tfidf_matrix)[0]

# Sentence embedding scores for long docs
corpus_embs = model.encode(long_docs, normalize_embeddings=True)
query_emb   = model.encode([query],   normalize_embeddings=True)
emb_scores  = cosine_similarity(query_emb, corpus_embs)[0]

# Combined ranking by raw score — BUG: scores come from different spaces
all_scores = list(zip(short_docs + long_docs,
                      list(tfidf_scores) + list(emb_scores)))
all_scores.sort(key=lambda x: x[1], reverse=True)
print("Combined ranking:")
for doc, score in all_scores:
    print(f"  {score:.4f}  {doc[:70]}")


---
## Review Question 1: Are TF-IDF and embedding scores comparable?

Print the score range for TF-IDF and for sentence embeddings on the same query. Are they on the same scale? What happens when you rank them together?

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Not comparable:** TF-IDF cosine scores for short documents depend on vocabulary overlap; for sparse short docs they can range 0.0–1.0 but often cluster near 0 for unrelated pairs. Sentence embedding cosine scores with `normalize_embeddings=True` also produce values in [0, 1], but the absolute values reflect semantic proximity in a dense space trained on large corpora — even unrelated sentences typically score 0.2–0.4 due to shared subword structure.

**Why combining them breaks ranking:** The two scores are calibrated differently. A TF-IDF score of 0.5 for "FastAPI install" (exact keyword match) and an embedding score of 0.5 for a long semantically related document are not equivalent. Naively sorting by raw score biases results toward whichever modality produces higher absolute values on your data. A fair hybrid needs per-modality min-max normalisation, reciprocal rank fusion, or a learned combination weight.

</details>

In [ ]:
print(f"TF-IDF scores:     min={tfidf_scores.min():.4f}  max={tfidf_scores.max():.4f}")
print(f"Embedding scores:  min={emb_scores.min():.4f}   max={emb_scores.max():.4f}")
# Explain: are these comparable? What would a fair combination strategy look like?


---
## Review Question 2: Sentence embeddings and word order

Are `"Python installation error"` and `"error installation Python"` identical under sentence embeddings? They shouldn't be — demonstrate this.

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Not identical:** Transformer-based encoders process tokens through attention layers that are position-aware. "Python installation error" and "error installation Python" produce subtly different embeddings (cosine similarity slightly below 1.0, not 0.9999…). The model weights sequences based on context, so word order does matter — though for short, keyword-like strings the difference is small.

**GloVe-style contrast:** Averaged word embeddings (GloVe, fastText mean pooling) produce identical vectors for the same bag of words regardless of order, because there is no positional encoding or attention. This is a meaningful architectural difference: for NLP tasks where word order matters (negation, syntactic role), transformer encoders are strictly more expressive.

</details>

In [ ]:
a = "Python installation error"
b = "error installation Python"
emb_a = model.encode([a], normalize_embeddings=True)
emb_b = model.encode([b], normalize_embeddings=True)
sim = cosine_similarity(emb_a, emb_b)[0][0]
print(f"Cosine similarity: {sim:.4f}")
print("Are they identical embeddings?", np.allclose(emb_a, emb_b))
# If they were averaged word embeddings (GloVe-style), what would the similarity be?


---
## Review Question 3: Out-of-vocabulary handling for 'FastAPI'

If the corpus contains `"FastAPI install"`, how does `all-MiniLM-L6-v2` handle the token `"FastAPI"` vs a GloVe-style fixed vocabulary model? Demonstrate by checking TF-IDF vs embedding similarity for a FastAPI query.

<details>
<summary>🔑 Reveal answer — Q3</summary>

**TF-IDF:** Maps tokens to vocabulary indices at fit time. "FastAPI" not seen during fitting → its TF-IDF weight is 0 → the query vector ignores this token entirely. Any document containing "FastAPI" is invisible to the query from a term-matching perspective; only the remaining overlapping tokens (e.g., "web", "framework") contribute.

**Sentence embeddings:** `all-MiniLM-L6-v2` uses WordPiece subword tokenisation. "FastAPI" is split into sub-units that exist in the pretrained vocabulary (e.g., "Fast", "##API"). The model generates a meaningful contextual embedding even for unseen compound words. The demonstration shows embedding similarity correctly retrieves "FastAPI install guide" even when TF-IDF returns 0 non-zero features for the query.

</details>

In [ ]:
fastapi_query = "FastAPI web framework"
docs_with_fastapi = ["FastAPI install guide", "flask web application"]

# TF-IDF — what happens if 'FastAPI' is OOV in the fitted vocabulary?
tv2 = TfidfVectorizer()
tv2.fit(["flask web application"])  # fit without FastAPI
q_vec = tv2.transform([fastapi_query])
print("'fastapi' in TF-IDF vocab:", 'fastapi' in tv2.vocabulary_)
print("Non-zero features for query:", q_vec.nnz)

# Sentence embeddings — subword tokenisation handles OOV
emb_docs  = model.encode(docs_with_fastapi, normalize_embeddings=True)
emb_query = model.encode([fastapi_query],   normalize_embeddings=True)
sims = cosine_similarity(emb_query, emb_docs)[0]
for doc, s in zip(docs_with_fastapi, sims):
    print(f"  {s:.4f}  {doc}")


---
## Review Question 4: Model selection for asymmetric retrieval

`all-MiniLM-L6-v2` vs `multi-qa-MiniLM-L6-cos-v1` — what is the difference for asymmetric retrieval where the query is short and documents are long?

<details>
<summary>🔑 Reveal answer — Q4</summary>

**`all-MiniLM-L6-v2`** was trained on symmetric tasks (sentence-pair similarity, NLI) where both inputs are at the same granularity. When the query is a short natural-language question and documents are long descriptive paragraphs, the model may not optimally align the representations.

**`multi-qa-MiniLM-L6-cos-v1`** was trained specifically on question-answer pairs (asymmetric retrieval) — short queries against longer documents. It learns to place short question embeddings near the embeddings of long answers that address them, producing better nDCG on retrieval benchmarks. The practical difference shows up most on out-of-domain queries where the symmetric model conflates query and document semantics. Use `multi-qa-*` for search/FAQ retrieval; use `all-MiniLM-*` for symmetric duplicate detection or clustering.

</details>

In [ ]:
# Load the retrieval-optimised model and compare
try:
    model_qa = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
    long_doc = ("Sentence transformers encode variable-length text into fixed-size dense vectors "
                "using a pretrained transformer with mean pooling over the final hidden states.")
    short_query = "how do sentence transformers work"
    emb_gen = model.encode([short_query, long_doc],    normalize_embeddings=True)
    emb_qa  = model_qa.encode([short_query, long_doc], normalize_embeddings=True)
    print(f"all-MiniLM similarity:    {cosine_similarity([emb_gen[0]], [emb_gen[1]])[0][0]:.4f}")
    print(f"multi-qa-MiniLM similarity: {cosine_similarity([emb_qa[0]], [emb_qa[1]])[0][0]:.4f}")
except Exception as e:
    print(f"Model load error: {e}")
# When would you choose one over the other?


---
## Review Question 5: Sentence embeddings on very short documents

For 2–3 word short docs like `"Python error"`, what do you lose by using sentence embeddings vs TF-IDF? When is TF-IDF actually the better choice for short queries?

<details>
<summary>🔑 Reveal answer — Q5</summary>

**What you lose with embeddings on short docs:** A 2–3 word query like "Python error" is encoded by mean-pooling just a handful of token embeddings. The resulting vector is less stable and may not distinguish between "Python error", "Python install", or "Python tutorial" — the semantic signal is too weak. Transformer attention also has less context to work with.

**When TF-IDF is better:** For short, precise keyword queries (a library name, an error type, a product code), exact token matching is often exactly what you want — "FastAPI" should retrieve documents containing "FastAPI", not documents about generic web frameworks. TF-IDF handles this perfectly with zero model overhead. A practical hybrid: use TF-IDF for queries under 4–5 tokens where lexical precision matters, and sentence embeddings for longer natural-language questions where paraphrase and semantic matching are valuable.

</details>

In [ ]:
short_queries = ["Python error", "pandas merge", "install"]
long_query    = "I am getting a Python AttributeError when I try to merge two pandas dataframes"

for q in short_queries + [long_query]:
    emb = model.encode([q], normalize_embeddings=True)
    sims = cosine_similarity(emb, corpus_embs)[0]
    best = long_docs[np.argmax(sims)]
    print(f"Query: '{q[:50]}'")
    print(f"  Best match: '{best[:70]}'  sim={sims.max():.4f}")
    print()
# Explain: for which query lengths/types does TF-IDF outperform sentence embeddings?
